In [ ]:
# ============================================================
# ENGINEERING MEASUREMENT UNCERTAINTIES - GUM CALC  
# ============================================================

import sys

sys.path.insert(0, r"C:\Users\victo\Desktop\Python\gum-calc")

try:
    from gum_calc import (
        uncertainty_type_A,
        uncertainty_type_B_from_resolution,
        uncertainty_type_B_uniform,
        uncertainty_type_B_relative,
        uncertainty_type_exact,
        UncertaintyInput,
        linear_regression,
        nonlinear_regression,
        full_gum_analysis,
        generate_bilan,
        generate_bilan_regression,
        generate_bilan_nonlinear_regression,
        generate_bilan_compatibilite,
        generate_annexe,
        full_pipeline_regression_to_measurand,
    )
except ModuleNotFoundError as e:
    raise ModuleNotFoundError(
        "gum_calc introuvable : vérifiez que le chemin passé à "
        "sys.path.insert ci-dessus pointe bien vers le dossier qui "
        "contient gum_calc.py sur cette machine."
    ) from e

# UncertaintyInput est une dataclass — accès par attribut UNIQUEMENT :
#   u_X.u    → incertitude-type
#   u_X.nu   → degrés de liberté
#   u_X.N    → nombre de mesures (type A seulement)
#   u_X.s    → écart-type empirique (type A seulement)
#   u_X.mean → moyenne empirique (type A seulement)
# NE PAS écrire u_X["u"] ou u_X["mean"] → TypeError garanti

print("gum_calc chargé avec succès.")

In [ ]:
# ============================================================
# MESURANDE — Exemple : Résistance (R), à dupliquer/adapter pour
# chaque grandeur du TP (en renommant systématiquement le suffixe
# _R par le suffixe de la grandeur traitée dans la cellule copiée).
# ============================================================

# --- Sources d'incertitude : NE GARDER QU'UNE SEULE ligne par variable,
# selon la nature de la mesure ; les quatre autres restent en commentaire.

# Type A : nommer la liste pour pouvoir en extraire la moyenne via .mean
U_values = [5.02, 5.01, 5.03, 5.02, 5.00]   # N = 5 mesures répétées
u_U      = uncertainty_type_A(U_values)       # u_U.mean est synchronisé avec la liste
# u_U = uncertainty_type_B_from_resolution(resolution=0.01)   # Type B : résolution instrument
# u_U = uncertainty_type_B_uniform(half_width=0.01)           # Type B : demi-largeur connue
# u_U = uncertainty_type_B_relative(u_standard=0.01)          # Type B : u connue (notice)
# u_U = uncertainty_type_exact()                               # Constante exacte

u_I = uncertainty_type_B_from_resolution(resolution=0.001)   # Type B : résolution instrument
# u_I = uncertainty_type_A([...])
# u_I = uncertainty_type_B_uniform(half_width=...)
# u_I = uncertainty_type_B_relative(u_standard=...)
# u_I = uncertainty_type_exact()

# --- Valeurs nominales ---
# Règle : si type A, utiliser .mean (synchronisé) — jamais un littéral codé en dur.
#         si type B, la valeur nominale est fournie par la mesure directe (OK codée en dur).
nominales_R = {
    "U": u_U.mean,   # ← toujours synchronisé avec U_values
    "I": 0.502,      # ← type B : valeur nominale externe, OK
}

# --- Incertitudes ---
incertitudes_R = {
    "U": u_U,
    "I": u_I,
}

# --- Analyse GUM ---
res_R = full_gum_analysis(
    formula_str        = "U / I",
    variable_names     = ["U", "I"],
    nominal_values     = nominales_R,
    uncertainty_inputs = incertitudes_R,
)

# --- Bilan LaTeX : _res_precomputed réutilise le calcul ci-dessus
# plutôt que de relancer toute l'analyse GUM une seconde fois ---
bilan_R = generate_bilan(
    measurand_name     = "Résistance",
    measurand_symbol   = "R",
    formula_str        = "U / I",
    variable_names     = ["U", "I"],
    variable_symbols   = {"U": "U", "I": "I"},
    variable_units     = {"U": r"\volt", "I": r"\ampere"},
    nominal_values     = nominales_R,
    uncertainty_inputs = incertitudes_R,
    measurand_unit     = r"\ohm",
    _res_precomputed   = res_R,
)

# --- Affichage console ---
print(f"R = {res_R['result_rounded']} ± {res_R['U_rounded']}")
print(f"uc = {res_R['uc']:.4g}  |  k = {res_R['k']:.3f}  |  ν_eff = {res_R['nu_eff']:.1f}")
print("Budget :", {k: f"{v:.1f}%" for k, v in res_R['budget'].items()})

In [ ]:
# ============================================================
# COMPATIBILITÉ THÉORIE / MESURE — Exemple : Résistance (R)
# ============================================================
#
# Ce bloc répond à une question différente du bilan GUM ci-dessus :
# non pas "quelle est l'incertitude de R ?" mais "R mesurée est-elle
# compatible avec une valeur théorique de référence ?". Placement
# volontairement dans la cellule de la grandeur concernée, pas dans
# le bloc annexe : generate_bilan_compatibilite ne rentre PAS dans
# generate_annexe, qui reste réservé aux bilans generate_bilan.

# --- Cas mesurande direct : uc et nu_eff viennent de full_gum_analysis ---
R_theorique = 10.0   # valeur théorique de référence attendue pour R, à adapter

compat_R = generate_bilan_compatibilite(
    measurand_symbol = "R",
    measurand_name   = "Résistance",
    y_mesure         = res_R["result"],
    uc               = res_R["uc"],
    nu_eff           = res_R["nu_eff"],
    y_theorique      = R_theorique,
    measurand_unit   = r"\ohm",
)

print(compat_R)

# --- Cas paramètre de régression (à décommenter/adapter si la grandeur
# confrontée à la théorie est une pente ou une ordonnée à l'origine issue
# de linear_regression, plutôt qu'un mesurande direct) :
#
# reg = linear_regression(x_data, y_data)
# compat_theta1 = generate_bilan_compatibilite(
#     measurand_symbol = r"\theta_1",
#     y_mesure         = reg["theta1"],
#     uc               = reg["u_theta1"],
#     nu_eff           = reg["nu"],
#     y_theorique      = <valeur_theorique_pente>,
#     measurand_unit   = r"<unité>",
# )
# print(compat_theta1)

In [ ]:
# ============================================================
# MESURANDE — Exemple : régression + mesurande dérivé (Vitesse V
# à partir d'une régression Delta_T = theta0 + theta1 * D), à
# dupliquer/adapter si le TP contient une grandeur obtenue par
# régression linéaire plutôt que par mesure directe.
# ============================================================

# --- Données brutes de la régression ---
D_data  = [180, 210, 240, 270, 300, 330]   # cm, variable x
dT_data = [16, 18, 20, 22, 24.6, 25.6]     # ns, variable y

# ------------------------------------------------------------------
# PIEGE N.1 (voir SKILL.md 3.4ter-a) : full_pipeline_regression_to_
# measurand N'INJECTE PAS automatiquement theta1/theta0 dans les
# valeurs nominales et incertitudes du mesurande derive. Il FAUT
# passer couple_theta1=True des que formula_str contient "theta1",
# et couple_theta0=True des que formula_str contient "theta0".
# Omettre le flag alors que la formule reference theta1/theta0 leve :
#   ValueError: Valeur nominale manquante pour : ['theta1']
#
# PIEGE N.2 (voir SKILL.md 3.4ter-b) : linear_regression(x, y) ajuste
# TOUJOURS y = theta0 + theta1 * x, donc theta1 est en (unite de y)
# / (unite de x) -- jamais l'inverse. Si le mesurande depend en
# realite de x/y plutot que de y/x, utiliser 1/theta1 dans formula_str,
# pas theta1 directement. Ce piege NE LEVE AUCUNE ERREUR a l'execution :
# le calcul aboutit avec un resultat faux en ordre de grandeur mais
# plausible. Verifier systematiquement par analyse dimensionnelle
# avant d'ecrire formula_str : unite(theta1) = unite(y_data)/unite(x_data).
# ------------------------------------------------------------------

# Ici : Delta_T (ns) = theta0 + theta1 * D (cm) => theta1 en ns/cm.
# Le mesurande V = 2D/Delta_T (vitesse) depend de l'INVERSE de theta1 :
# formula_str = "2 * 1e7 / theta1"  (1e7 = facteur de conversion
# cm/ns -> m/s), et surtout PAS "2 * theta1 * 1e7".

bilan_V = full_pipeline_regression_to_measurand(
    x_data=D_data, y_data=dT_data,
    x_symbol="D", y_symbol=r"\Delta T",
    x_unit=r"\centi\meter", y_unit=r"\nano\second",
    formula_str="2 * 1e7 / theta1",
    variable_names=["theta1"],
    variable_symbols={"theta1": r"a"},
    variable_units={"theta1": r"\nano\second\per\centi\meter"},
    nominal_values_helpers={},
    uncertainty_inputs_helpers={},
    measurand_symbol="V",
    measurand_unit=r"\meter\per\second",
    slope_unit=r"\nano\second\per\centi\meter",
    intercept_unit=r"\nano\second",
    slope_symbol=r"\theta_1",
    intercept_symbol=r"\theta_0",
    couple_theta1=True,   # OBLIGATOIRE : formula_str reference theta1
)

print(bilan_V)

In [ ]:
# ============================================================
# COMPATIBILITÉ THÉORIE / MESURE — Exemple régression : Vitesse (V)
# ============================================================
#
# full_pipeline_regression_to_measurand ne renvoie que du LaTeX,
# jamais les valeurs numeriques du mesurande derive. Pour le test
# de compatibilite sur V (et non sur theta1 seul), on rejoue donc
# la meme chaine regression -> full_gum_analysis, avec exactement
# la meme construction de theta1 (type B, nu = reg["nu"]) que celle
# appliquee en interne par full_pipeline_regression_to_measurand
# lorsque couple_theta1=True. Aucune propagation n'est recalculee
# manuellement : on ne fait que reconstruire les entrees.

reg_V = linear_regression(D_data, dT_data)

nominales_V = {"theta1": reg_V["theta1"]}
incertitudes_V = {
    "theta1": UncertaintyInput(
        u=reg_V["u_theta1"], type="B", distribution="general", nu=reg_V["nu"]
    )
}

res_V = full_gum_analysis(
    formula_str        = "2 * 1e7 / theta1",
    variable_names     = ["theta1"],
    nominal_values     = nominales_V,
    uncertainty_inputs = incertitudes_V,
)

V_theorique = 2.99792458e8   # m/s, exemple : valeur exacte (definition du metre)

compat_V = generate_bilan_compatibilite(
    measurand_symbol = "V",
    measurand_name   = "Vitesse",
    y_mesure         = res_V["result"],
    uc               = res_V["uc"],
    nu_eff           = res_V["nu_eff"],
    y_theorique      = V_theorique,
    measurand_unit   = r"\meter\per\second",
)

print(compat_V)
# NE PAS ajouter compat_V a la liste passee a generate_annexe : ce
# bloc reste dans sa propre cellule, jamais dans le bloc annexe final.

In [ ]:
# ============================================================
# RÉGRESSION NON LINÉAIRE — Exemple : décharge RC, V(t) = V0 * exp(-t/tau)
# à dupliquer/adapter si le TP contient une grandeur obtenue par un
# ajustement non affine (exponentielle, sinusoïde, loi de puissance...)
# plutôt que par une régression linéaire ou une mesure directe.
# ============================================================

import numpy as np

# ------------------------------------------------------------------
# PIÈGE — model_func DOIT utiliser des opérations numpy (np.exp, np.sin,
# np.sqrt...), jamais le module math : nonlinear_regression appelle
# model_func sur un tableau numpy (x_data entier), pas point par point.
# Utiliser math.exp au lieu de np.exp lève un TypeError explicite à
# l'exécution — jamais un résultat silencieusement faux.
# ------------------------------------------------------------------

def modele_decharge_RC(t, V0, tau):
    return V0 * np.exp(-t / tau)

# --- Données brutes de l'ajustement ---
t_data = [0.0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0, 8.0]   # ms
V_data = [5.01, 3.94, 3.06, 2.42, 1.89, 1.48, 1.18, 0.90, 0.72, 0.46, 0.29, 0.12]  # V

# --- Estimation initiale p0 : la cause la plus fréquente de non-
# convergence de curve_fit est un p0 trop éloigné de la solution.
# Lire p0 directement sur le nuage de points avant de lancer l'ajustement
# (ici : V0 ≈ V(t=0), tau ≈ temps pour atteindre V0/e) plutôt que de
# deviner au hasard. ---
p0_RC = [5.0, 2.0]

reg_RC = nonlinear_regression(
    x_data       = t_data,
    y_data       = V_data,
    model_func   = modele_decharge_RC,
    p0           = p0_RC,
    param_names  = ["V0", "tau"],
    # sigma      = None,   # incertitude-type sur V connue point par point (résolution
                            # du voltmètre par ex.) → décommenter et renseigner, sinon
                            # l'incertitude sur V0/tau est estimée depuis la dispersion
                            # des résidus eux-mêmes (non pondéré, cf. docstring)
)

# --- Bilan LaTeX : _reg_precomputed réutilise l'ajustement ci-dessus
# plutôt que de relancer curve_fit une seconde fois ---
bilan_RC = generate_bilan_nonlinear_regression(
    x_symbol         = "t",
    y_symbol         = "V",
    x_unit           = r"\milli\second",
    y_unit           = r"\volt",
    x_data           = t_data,
    y_data           = V_data,
    model_func       = modele_decharge_RC,
    p0               = p0_RC,
    param_names      = ["V0", "tau"],
    param_symbols    = {"V0": "V_0", "tau": r"\tau"},
    param_units      = {"V0": r"\volt", "tau": r"\milli\second"},
    # model_latex : UN '{}' positionnel PAR PARAMÈTRE de param_names,
    # dans le MÊME ordre (ici V0 puis tau) — voir 3.4quater du skill.
    model_latex      = r"{} \exp\left(-t/{}\right)",
    subsection_title = "Décharge RC",
    _reg_precomputed = reg_RC,
)

# --- Affichage console ---
print(f"V0  = {reg_RC['params']['V0']:.4g} ± {reg_RC['u_params']['V0']:.2g} V")
print(f"tau = {reg_RC['params']['tau']:.4g} ± {reg_RC['u_params']['tau']:.2g} ms")
print(f"nu = {reg_RC['nu']}  |  r² = {reg_RC['r2']:.4f}")

# --- Chaînage éventuel : si un mesurande ultérieur dépend de tau (ex :
# une constante RC = tau * facteur), propager EXACTEMENT comme pour un
# paramètre de régression linéaire (cf. section 3.5 du skill) :
#
# u_tau_chaine = uncertainty_type_B_relative(u_standard=reg_RC["u_params"]["tau"])
# u_tau_chaine.nu = reg_RC["nu"]   # OBLIGATOIRE — sinon nu retombe à l'infini par défaut
#
# NE JAMAIS utiliser directement reg_RC["cov"] à la main dans une chaîne
# — ce n'est utile QUE si le mesurande combine V0 ET tau dans la MÊME
# formule (paramètre `covariances` de full_gum_analysis/generate_bilan),
# pas pour une propagation en cascade vers un mesurande indépendant.

In [ ]:
# ============================================================
# GÉNÉRATION DU LATEX
# ============================================================

print(generate_annexe([
    bilan_R,
    # bilan_<G2>,  # ajouter ici un bilan_<G> par mesurande de la cellule précédente, dans l'ordre du CR
]))